# Визуально-языковая LMM в каскадной архитектуре детекции контрафакта

## Что проверяется в данном ноуте

В ноутах 03 (zero-shot CLIP, негативный результат) и 04 (zero-shot Qwen2.5-1.5B-Instruct + LoRA fine-tune, также негативный) был зафиксирован двойной отрицательный результат применения foundation-моделей в zero-shot и малоёмком LoRA-режимах к маркетплейс-задаче. В обоих случаях Stage 2 каскадной архитектуры (см. § 4.4.9 диплома) работал **унимодально**: CLIP — только на изображении, Qwen2.5 — только на структурированном текстовом представлении товара.

В настоящем ноуте проверяется применимость **подлинно мультимодальной** large multimodal model (LMM) — **Qwen2-VL-2B-Instruct** (Wang et al., 2024, arXiv:2409.12191), которая в единственном forward-проходе совместно обрабатывает raw-изображение товара и его структурированное текстовое описание. Это закрывает аргумент о том, что наблюдаемая ограниченность Qwen2.5-1.5B-Instruct (текстовой LLM) объясняется отсутствием визуальной модальности в Stage 2.

## Краткий обзор связанной литературы

В открытой академической литературе на момент написания работы (май 2026 г.) отсутствуют peer-reviewed эмпирические эксперименты с применением визуально-языковой LMM (Qwen2-VL, LLaVA-OneVision, Anomaly-OneVision) к задаче детекции контрафакта на маркетплейсе с публичной квантитативной оценкой ROC-AUC и PR-AUC. Ближайшие смежные направления (подробный обзор см. в § 1.5.4.6 диплома):

- **Anomaly-OneVision** (Xu et al., CVPR 2025, arXiv:2502.07601) — первый специализированный визуальный ассистент для zero-shot anomaly detection на базе LLaVA-OneVision. Image-level AUROC 88,6 % на индустриальных дефектах; механизм Look-Twice Feature Matching.
- **FIDAVL** (Keita et al., arXiv:2409.03109) — VLM для детекции синтетических изображений (95 %+ accuracy).
- **Vision-Language Models for E-commerce Non-Compliant** (Vajda et al., ICAART 2025) — VLM для compliance-проверки маркетплейс-карточек (не контрафакт).

С отраслевой стороны Amazon в 2024 году публично заявила о внедрении multimodal LLMs для холистического анализа карточек, что позволяет блокировать более 99 % подозрительных подделок ([Amazon 2024 Brand Protection Report](https://trustworthyshopping.aboutamazon.com/2024-brand-protection-report)), однако архитектурные детали в открытых источниках отсутствуют. Настоящий ноут закрывает академическую нишу для нашей задачи.

## Архитектура каскада с Qwen2-VL-2B как Stage 2

1. **Stage 1:** CatBoost-скрининг на табличных + K-means structural признаках (тот же, что в ноутах 03 и 04). Latency ≈ 5 мс на CPU.
2. **Stage 2 (новое):** Qwen2-VL-2B-Instruct в zero-shot режиме применяется к borderline-объектам со структурированным промптом, включающим: raw-изображение карточки + текстовые поля (название, бренд, цена, возраст товара/продавца, доля возвратов). Модель возвращает JSON `{"score", "reason"}`.
3. **Score fusion + сравнение** с Stage 1, zero-shot CLIP (ноут 03) и zero-shot Qwen2.5 (ноут 04).

## Параметры эксперимента

- Качественная демонстрация и квантитативная оценка на стратифицированном сэмпле из ~200 borderline-объектов test (25 % positive).
- Stage 1 borderline-зона: `tau_lo = 0,10`, `tau_hi = 0,85`.
- Qwen2-VL-2B-Instruct: 2,21 млрд параметров, fp16 на Apple M4 Pro с MPS-ускорением, расчётная латентность ~3–6 секунд на инференс одного объекта.

## Предварительные условия для запуска

- Установлены пакеты `transformers ≥ 5.x`, `torch ≥ 2.x` с MPS-поддержкой, `qwen-vl-utils`, `torchvision`, `Pillow`.
- Модель Qwen2-VL-2B-Instruct скачана в HuggingFace-cache (либо в `/tmp/qwen2vl_weights/`).
- Папка `train_images/` содержит raw-изображения, проиндексированные по `ItemID`. По умолчанию ожидается путь `data/ml_ozon_counterfeit_train_images/` (переопределяется через переменную `TRAIN_IMG_DIR`).

> **Связь с дипломом:** результаты настоящего ноута дают эмпирическое подтверждение или опровержение применимости визуально-языковой LMM как Stage 2 каскадной архитектуры и являются частью § 4.4.9 диплома.

## 1. Setup

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import gc, json, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from catboost import CatBoostClassifier
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

ROOT = Path('.')
DATA_PATH = ROOT / "data" / 'ml_ozon_ounterfeit_train.csv'
TRAIN_IMG_DIR = Path(os.environ.get(
    'TRAIN_IMG_DIR', str(ROOT / 'data' / 'ml_ozon_counterfeit_train_images')))
OUT_DIR = ROOT / 'new_diploma' / 'real_estate_approaches' / 'artifacts_qwen2vl'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'resolution'
CAT = 'CommercialTypeName4'

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Train images directory: {TRAIN_IMG_DIR}')
print(f'Train images directory exists: {TRAIN_IMG_DIR.exists()}')

## 2. Данные + seller-based split (как в ноуте 04)

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='utf-8')
print(f'Loaded: {df.shape}')

seller_targets = df.groupby('SellerID')[TARGET].max().reset_index()
train_sellers, temp_sellers = train_test_split(
    seller_targets['SellerID'], test_size=0.30,
    random_state=SEED, stratify=seller_targets[TARGET])
temp_targets = seller_targets[seller_targets['SellerID'].isin(temp_sellers)]
val_sellers, test_sellers = train_test_split(
    temp_targets['SellerID'], test_size=0.50,
    random_state=SEED, stratify=temp_targets[TARGET])
train_df = df[df['SellerID'].isin(train_sellers)].copy().reset_index(drop=True)
val_df = df[df['SellerID'].isin(val_sellers)].copy().reset_index(drop=True)
test_df = df[df['SellerID'].isin(test_sellers)].copy().reset_index(drop=True)
y_train, y_val, y_test = train_df[TARGET].to_numpy(), val_df[TARGET].to_numpy(), test_df[TARGET].to_numpy()
print(f'Train {len(train_df)}, Val {len(val_df)}, Test {len(test_df)}')
print(f'Positive rates — Train {y_train.mean():.4f}, Val {y_val.mean():.4f}, Test {y_test.mean():.4f}')

## 3. Проверка покрытия изображениями

Для квантитативной оценки на borderline-test необходимо, чтобы соответствующие raw-изображения присутствовали в `TRAIN_IMG_DIR`. Ниже подсчитывается покрытие.

In [ ]:
if not TRAIN_IMG_DIR.exists():
    print(f'WARN: {TRAIN_IMG_DIR} does not exist yet. Skip image-coverage check.')
    img_ids = set()
else:
    img_ids = set()
    for ext in ('*.png', '*.jpg', '*.jpeg'):
        for f in TRAIN_IMG_DIR.glob(ext):
            try:
                img_ids.add(int(f.stem))
            except ValueError:
                pass
    print(f'Total image files on disk: {len(img_ids):,}')
    in_train = sum(int(i) in img_ids for i in train_df.ItemID)
    in_val = sum(int(i) in img_ids for i in val_df.ItemID)
    in_test = sum(int(i) in img_ids for i in test_df.ItemID)
    print(f'Train items with image: {in_train:,} of {len(train_df):,} ({in_train/len(train_df)*100:.1f}%)')
    print(f'Val   items with image: {in_val:,} of {len(val_df):,} ({in_val/len(val_df)*100:.1f}%)')
    print(f'Test  items with image: {in_test:,} of {len(test_df):,} ({in_test/len(test_df)*100:.1f}%)')

## 4. Stage 1 (CatBoost), идентичный ноутам 03/04

In [ ]:
refs = {
    'cat_med': train_df.groupby(CAT)['PriceDiscounted'].median(),
    'cat_tgt': train_df.groupby(CAT)[TARGET].mean(),
    'brand_tgt': train_df.groupby('brand_name')[TARGET].mean(),
    'global_med': train_df['PriceDiscounted'].median(),
    'global_tgt': train_df[TARGET].mean(),
}

def engineer(frame):
    out = frame.copy()
    cm = out[CAT].map(refs['cat_med']).fillna(refs['global_med'])
    out['price_ratio'] = out['PriceDiscounted'].fillna(0) / cm.replace(0, np.nan).fillna(1)
    out['category_target_mean'] = out[CAT].map(refs['cat_tgt']).fillna(refs['global_tgt'])
    out['brand_target_mean'] = out['brand_name'].map(refs['brand_tgt']).fillna(refs['global_tgt'])
    out['return_rate_30'] = out['item_count_returns30'].fillna(0) / (out['item_count_sales30'].fillna(0) + 1)
    out['return_rate_90'] = out['item_count_returns90'].fillna(0) / (out['item_count_sales90'].fillna(0) + 1)
    out['is_new_item'] = (out['item_time_alive'].fillna(0) <= 30).astype(int)
    out['is_new_seller'] = (out['seller_time_alive'].fillna(0) <= 180).astype(int)
    return out

train_df, val_df, test_df = engineer(train_df), engineer(val_df), engineer(test_df)

tab_feats = [
    'PriceDiscounted', 'item_time_alive', 'seller_time_alive',
    'item_count_sales30', 'item_count_sales90',
    'item_count_returns30', 'item_count_returns90',
    'GmvTotal30', 'GmvTotal90',
    'price_ratio', 'category_target_mean', 'brand_target_mean',
    'return_rate_30', 'return_rate_90',
    'is_new_item', 'is_new_seller',
]
sc = StandardScaler()
Xtr = sc.fit_transform(train_df[tab_feats].fillna(0))
Xv = sc.transform(val_df[tab_feats].fillna(0))
Xte = sc.transform(test_df[tab_feats].fillna(0))

km = KMeans(n_clusters=2, random_state=SEED, n_init=10)
ctr = km.fit_predict(Xtr)
fc = int(pd.Series(y_train).groupby(ctr).mean().idxmax())
fcen = km.cluster_centers_[fc]
for d, X in [(train_df, Xtr), (val_df, Xv), (test_df, Xte)]:
    d['dist_centroid'] = np.linalg.norm(X - fcen, axis=1)
tab_feats.append('dist_centroid')
Xtr = sc.fit_transform(train_df[tab_feats].fillna(0))
Xv = sc.transform(val_df[tab_feats].fillna(0))
Xte = sc.transform(test_df[tab_feats].fillna(0))

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
stage1 = CatBoostClassifier(
    iterations=1000, depth=6, learning_rate=0.05,
    scale_pos_weight=scale_pos, random_seed=SEED,
    early_stopping_rounds=80, verbose=False,
)
stage1.fit(Xtr, y_train, eval_set=(Xv, y_val), use_best_model=True)
iso = IsotonicRegression(out_of_bounds='clip')
p_s1_val = stage1.predict_proba(Xv)[:, 1]
iso.fit(p_s1_val, y_val)
p_s1_test = iso.transform(stage1.predict_proba(Xte)[:, 1])

roc1 = roc_auc_score(y_test, p_s1_test)
pr1 = average_precision_score(y_test, p_s1_test)
print(f'Stage 1 on full test: ROC-AUC = {roc1:.4f}, PR-AUC = {pr1:.4f}')

## 5. Borderline-выборка для Qwen2-VL

Stage 2 (Qwen2-VL) дорогой по латентности (3–6 сек/объект), поэтому применяется только к объектам, где Stage 1 даёт оценку в borderline-зоне `[τ_lo, τ_hi]`. Дополнительно отфильтровываем по наличию изображения на диске.

In [ ]:
TAU_LO, TAU_HI = 0.10, 0.85
border_test_mask = (p_s1_test >= TAU_LO) & (p_s1_test <= TAU_HI)
border_test = np.where(border_test_mask)[0]
print(f'Borderline test: {len(border_test):,} объектов')

if img_ids:
    border_with_img = [i for i in border_test if int(test_df.iloc[int(i)].ItemID) in img_ids]
else:
    border_with_img = list(border_test)
print(f'Из них с изображением на диске: {len(border_with_img):,}')

rng = np.random.default_rng(SEED)
N_TEST = int(os.environ.get('N_TEST', '200'))
pos = [i for i in border_with_img if y_test[i] == 1]
neg = [i for i in border_with_img if y_test[i] == 0]
n_pos = min(len(pos), N_TEST // 4)
n_neg = N_TEST - n_pos
test_idx = np.concatenate([
    rng.choice(pos, size=n_pos, replace=False) if len(pos) >= n_pos else np.array(pos),
    rng.choice(neg, size=min(n_neg, len(neg)), replace=False),
])
rng.shuffle(test_idx)
print(f'Финальная подвыборка для Qwen2-VL: {len(test_idx)} объектов ({y_test[test_idx].mean()*100:.1f}% positive)')

## 6. Загрузка Qwen2-VL-2B-Instruct

In [ ]:
MODEL_NAME = 'Qwen/Qwen2-VL-2B-Instruct'
print(f'Loading {MODEL_NAME} on {device}...')
t0 = time.time()
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, dtype=torch.float16 if device == 'mps' else torch.float32)
model.to(device); model.eval()
# Ограничиваем число visual-токенов через min/max_pixels: при native-разрешении
# Qwen2-VL может выделять буфер attention в десятки ГБ — на M4 Pro это даёт OOM.
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,   # ~200k токенов снизу
    max_pixels=768 * 28 * 28,   # ~600k токенов сверху — комфортно укладывается в 24 ГБ MPS
)
print(f'Loaded in {time.time()-t0:.0f}s')
print(f'Trainable params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## 7. Промпт-инжиниринг для визуально-языкового reasoning

Промпт устроен следующим образом. Системная инструкция фиксирует роль (эксперт по детекции контрафакта на Ozon) и формат выхода (JSON со score и кратким объяснением). Пользовательский ввод содержит изображение товара и структурированный текстовый блок с табличными признаками. Модель в zero-shot режиме должна совместить визуальные паттерны (качество логотипа, упаковки, фото) и текстовые/числовые сигналы для вынесения суждения.

In [ ]:
SYSTEM = (
    "Ты эксперт по обнаружению контрафактных товаров на онлайн-маркетплейсе Ozon. "
    "По изображению и описанию товара оцени, является ли он контрафактным. "
    "Учитывай и визуальные признаки (качество логотипа, упаковки, фото, маркировка), "
    "и текстовые/числовые данные (бренд, цена, возраст товара и продавца, доля возвратов). "
    "Выдай строго JSON: "
    '{"score": <0.0..1.0>, "reason": "<краткое объяснение на русском>"}'
)

def _s(v, default=''):
    if v is None: return default
    if isinstance(v, float) and pd.isna(v): return default
    s = str(v); return s if s and s != 'nan' else default

def build_user_text(row):
    name = _s(row.get('name_rus'))[:100]
    brand = _s(row.get('brand_name'), default='—') or '—'
    cat = _s(row.get(CAT), default='—') or '—'
    price = row.get('PriceDiscounted', 0) or 0
    pr_r = row.get('price_ratio', 1.0)
    ia = row.get('item_time_alive', 0) or 0
    sa = row.get('seller_time_alive', 0) or 0
    rr = row.get('return_rate_30', 0)
    return (
        f'Название: "{name}"\nБренд: "{brand}"\nКатегория: "{cat}"\n'
        f'Цена: {price:.0f} руб (price_ratio = {pr_r:.2f})\n'
        f'Возраст товара: {ia:.0f} дней\nВозраст продавца: {sa:.0f} дней\n'
        f'Доля возвратов за 30 дней: {rr*100:.1f}%')

def parse_score(text):
    m = re.search(r'\{[^{}]*?"score"[^{}]*?\}', text, re.DOTALL)
    if not m: return None
    try:
        obj = json.loads(m.group(0))
        s = float(obj.get('score', 0.5))
        return max(0.0, min(1.0, s))
    except Exception:
        return None

@torch.no_grad()
def vl_score(image_path, user_text, max_new_tokens=80):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': user_text}]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt')
    inputs = inputs.to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = processor.batch_decode([out[0][inputs.input_ids.shape[1]:]],
        skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return answer

# Smoke test on 1 sample
if len(test_idx) > 0:
    idx0 = int(test_idx[0])
    row0 = test_df.iloc[idx0]
    item_id0 = int(row0.ItemID)
    img_path = TRAIN_IMG_DIR / f'{item_id0}.png'
    if not img_path.exists():
        img_path = TRAIN_IMG_DIR / f'{item_id0}.jpg'
    print(f'Smoke test on ItemID {item_id0} ({img_path.name})')
    print(f'User prompt:\n{build_user_text(row0)}')
    t0 = time.time()
    ans = vl_score(img_path, build_user_text(row0))
    print(f'\nLatency: {time.time()-t0:.1f}s')
    print(f'Raw answer: {ans}')
    print(f'Parsed score: {parse_score(ans)}')

## 8. Инференс Qwen2-VL на borderline-выборке

In [ ]:
scores = np.full(len(test_idx), np.nan)
fails = 0
log_rows = []
t0 = time.time()
for j, idx in enumerate(test_idx):
    row = test_df.iloc[int(idx)]
    item_id = int(row.ItemID)
    p = TRAIN_IMG_DIR / f'{item_id}.png'
    if not p.exists():
        p = TRAIN_IMG_DIR / f'{item_id}.jpg'
    try:
        ans = vl_score(p, build_user_text(row))
        s = parse_score(ans)
        if s is not None:
            scores[j] = s
        else:
            fails += 1
        log_rows.append({
            'idx': int(idx), 'item_id': item_id, 'y': int(y_test[idx]),
            'stage1': float(p_s1_test[idx]), 'score': s if s is not None else None,
            'name': str(row.get('name_rus', ''))[:80],
            'brand': str(row.get('brand_name', '')),
            'answer': ans[:300]})
    except Exception as e:
        fails += 1
        log_rows.append({
            'idx': int(idx), 'item_id': item_id, 'y': int(y_test[idx]),
            'stage1': float(p_s1_test[idx]), 'score': None,
            'name': str(row.get('name_rus', ''))[:80],
            'brand': str(row.get('brand_name', '')),
            'answer': f'ERROR: {str(e)[:200]}'})
    if (j + 1) % 20 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (j + 1) * (len(test_idx) - j - 1)
        print(f'  {j+1}/{len(test_idx)}, fail={fails}, elapsed={elapsed:.0f}s, eta={eta:.0f}s')
print(f'\nDone in {time.time()-t0:.0f}s, parse_fails={fails}')

med = np.nanmedian(scores) if not np.all(np.isnan(scores)) else 0.5
scores = np.where(np.isnan(scores), med, scores)

results_df = pd.DataFrame(log_rows)
results_df.to_csv(OUT_DIR / 'qwen2vl_results.csv', index=False)
print(f'Saved {OUT_DIR}/qwen2vl_results.csv')

## 9. Метрики и сравнение с baselines

In [ ]:
y_sub = y_test[test_idx]
p_s1_sub = p_s1_test[test_idx]

def rap(y, p, m=0.9):
    pr, rc, _ = precision_recall_curve(y, p)
    msk = pr >= m
    return float(rc[msk].max()) if msk.any() else 0.0

print('=== Метрики на borderline-сэмпле test (Qwen2-VL vs baselines) ===')
print(f'{"Модель":<55} {"ROC":>8} {"PR":>8} {"R@P":>8}')
for label, p in [
    ('Stage 1 (CatBoost on tab + K-means)', p_s1_sub),
    ('Qwen2-VL-2B-Instruct (vision + text, this run)', scores),
]:
    print(f'{label:<55} {roc_auc_score(y_sub, p):>8.4f} {average_precision_score(y_sub, p):>8.4f} {rap(y_sub, p):>8.4f}')

print()
print('Контрольные значения из предыдущих экспериментов (для справки):')
print(f'  Zero-shot CLIP (ноут 03):                  ROC = 0.4795, PR = 0.0589')
print(f'  Zero-shot Qwen2.5-1.5B text-only (ноут 04): ROC = 0.5066, PR = 0.2529')
print(f'  LoRA Qwen2.5-1.5B text-only (§ 4.4.9.4):    ROC = 0.5200, PR = 0.2581')

## 10. Score fusion с Stage 1

In [ ]:
# Подбор веса w_VL на grid; так как у нас val-сэмпл маленький, используем непосредственно
# borderline test для отображения тренда (только для визуализации, не для финальной оценки).
print(f'{"w_VL":>6} | {"ROC":>8} {"PR":>8} {"R@P":>8}')
for w in np.arange(0.0, 1.01, 0.10):
    fused = (1 - w) * p_s1_sub + w * scores
    print(f'{w:>6.2f} | {roc_auc_score(y_sub, fused):>8.4f} {average_precision_score(y_sub, fused):>8.4f} {rap(y_sub, fused):>8.4f}')

## 11. Качественные примеры

Покажем по 3 примера успешного и неудачного распознавания. Это будет материалом для **Приложения Б** диплома.

In [ ]:
results_df['score_filled'] = scores

print('=== 3 случая true-positive (LMM-score высокий, метка = контрафакт) ===\n')
tp = results_df[(results_df['y'] == 1) & (results_df['score_filled'] > 0.6)].sort_values('score_filled', ascending=False).head(3)
for _, row in tp.iterrows():
    print(f'ItemID {row.item_id} | brand: {row.brand} | name: {row["name"]}')
    print(f'  Stage1 = {row.stage1:.3f}, Qwen2-VL = {row.score_filled:.3f}, true = {row.y}')
    print(f'  Answer: {row.answer[:300]}\n')

print('\n=== 3 случая true-negative (LMM-score низкий, метка = оригинал) ===\n')
tn = results_df[(results_df['y'] == 0) & (results_df['score_filled'] < 0.3)].sort_values('score_filled').head(3)
for _, row in tn.iterrows():
    print(f'ItemID {row.item_id} | brand: {row.brand} | name: {row["name"]}')
    print(f'  Stage1 = {row.stage1:.3f}, Qwen2-VL = {row.score_filled:.3f}, true = {row.y}')
    print(f'  Answer: {row.answer[:300]}\n')

print('\n=== 3 случая false-positive (LMM ошибочно проставила высокий score) ===\n')
fp = results_df[(results_df['y'] == 0) & (results_df['score_filled'] > 0.6)].sort_values('score_filled', ascending=False).head(3)
for _, row in fp.iterrows():
    print(f'ItemID {row.item_id} | brand: {row.brand} | name: {row["name"]}')
    print(f'  Stage1 = {row.stage1:.3f}, Qwen2-VL = {row.score_filled:.3f}, true = {row.y}')
    print(f'  Answer: {row.answer[:300]}\n')

## 12. Сохранение артефактов

In [ ]:
np.save(OUT_DIR / 'test_vl_scores.npy', scores.astype(np.float32))
np.save(OUT_DIR / 'test_vl_indices.npy', test_idx)
print(f'Saved test_vl_scores.npy, test_vl_indices.npy')
print(f'Saved qwen2vl_results.csv с raw-ответами модели')
print(f'\nАртефакты в {OUT_DIR}')

## 13. Научные выводы

> **Эта ячейка должна быть заполнена после первого полного прогона ноута**, в зависимости от фактических метрик. Шаблон ниже учитывает три возможных исхода эксперимента и одинаково подойдёт ко всем (нужно выбрать подходящий и удалить остальные).

### Вариант A — ROC-AUC > 0,60 на borderline (положительный результат)

В отличие от двух предшествующих негативных результатов (zero-shot CLIP в § 4.4.9.2 с ROC = 0,48 и zero-shot Qwen2.5-1.5B текстовой LLM в § 4.4.9.3 с ROC = 0,51), визуально-языковая мультимодальная модель **Qwen2-VL-2B-Instruct** в zero-shot режиме на borderline-объектах маркетплейс-теста даёт ROC-AUC = <fill> и PR-AUC = <fill>. Это **первое в открытой академической литературе подтверждение применимости** визуально-языковой LMM к задаче детекции маркетплейс-контрафакта без какой-либо доменной адаптации. Стоит отметить, что прирост относительно унимодальных foundation-моделей объясняется именно совместной обработкой визуальных и текстовых сигналов в единственном forward-проходе: модель выявляет противоречия между визуальной семантикой товара и его текстовым представлением, что недоступно при раздельной обработке модальностей.

### Вариант B — ROC-AUC в диапазоне 0,55–0,60 (умеренный положительный результат)

Qwen2-VL-2B-Instruct в zero-shot режиме показывает ROC-AUC = <fill> и PR-AUC = <fill> на borderline-объектах, что **выше** zero-shot CLIP (ROC = 0,48) и zero-shot Qwen2.5-1.5B (ROC = 0,51), но **уступает** Stage 1 (CatBoost). Стоит отметить, что добавление настоящей мультимодальности (raw-изображение + текст в едином forward-проходе) даёт небольшой положительный сигнал по сравнению с унимодальными foundation-моделями, однако без supervised fine-tune этот сигнал недостаточен для опережения классической мультимодальной модели M2 с предвычисленными CLIP-эмбеддингами.

### Вариант C — ROC-AUC ≤ 0,55 (третий значимый негативный результат)

Qwen2-VL-2B-Instruct в zero-shot режиме демонстрирует ROC-AUC = <fill> и PR-AUC = <fill> на borderline-выборке, что находится в пределах статистической погрешности относительно zero-shot Qwen2.5-1.5B (ROC = 0,51). Этот результат является **третьим значимым научным негативным результатом** после § 4.4.9.2 (CLIP) и § 4.4.9.3 (Qwen2.5-text-only). Совокупно три эксперимента опровергают наивную гипотезу о применимости любых foundation-моделей в zero-shot режиме к маркетплейс-задаче без существенной доменной адаптации, **независимо от наличия визуальной модальности в модели**. Объяснение состоит в том, что фундаментальные сигналы контрафакта на маркетплейсе носят преимущественно поведенческий характер (продавец, его история, ценовое отклонение от категории), а визуально-семантическое сходство контрафакта и оригинала почти идеально (cosine между классовыми центроидами CLIP-пространства = 0,976, см. § 2.3.4), что не оставляет возможностей для zero-shot-различения через визуальные признаки.

### Связь с темой ВКР

Тема диплома формулируется как «...с использованием больших мультимодальных моделей». В настоящем ноуте применена **подлинно мультимодальная** foundation-модель Qwen2-VL-2B-Instruct, обрабатывающая изображение и текст в единственном forward-проходе. Этот эксперимент закрывает один из ключевых вопросов работы — **достаточно ли наличия визуальной модальности в LMM** для разрешения ограниченности унимодальных подходов (CLIP визуально и Qwen2.5 текстово). Полученный результат становится частью § 4.4.9 диплома и формирует основу для финальных рекомендаций по применению LVLM в production-сценариях.